In [ ]:
# Extract 6-hour AR labels for all lakes from the original AR data

import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path

root = Path('.')
ar_dataset = xr.open_dataset(root / 'globalARcatalog_ERA5_1940-2024_v4.0.nc')
ar_dataset = ar_dataset.chunk({"lat": 50, "lon": 50})
ar_time = ar_dataset['time'].to_numpy()

lake_info = pd.read_csv(root / 'Lake_info.csv', index_col=0)

lats = lake_info['lat'].tolist()
lons = lake_info['lon'].tolist()
lons = np.mod(lons, 360)

ar_dataset_sel = ar_dataset.sel(lat=xr.DataArray(lats, dims="points"),
                                lon=xr.DataArray(lons, dims="points"),
                                method="nearest")
shapemaps_sel = np.squeeze(ar_dataset_sel['shapemap']).compute().to_numpy()

ar_6h = pd.DataFrame(shapemaps_sel, columns=lake_info.index, index=ar_time)
ar_6h.to_csv(root / 'AR_extraction_6h.csv')

In [21]:
# Aggregate 6-hour AR occurrence data to daily AR presence/absence

ar_6h = pd.read_csv(root / 'AR_extraction_6h.csv', index_col=0)
ar_6h.index = pd.to_datetime(ar_6h.index)
ar_6h = ar_6h.loc['1950-09-01':'2022-08-31']
ar_time = ar_6h.index
ar_time_day = np.array(ar_time).reshape(-1, 4)[:,0]

ar_daily_dict = {}
for lakeid in ar_6h.columns:
    shapemap = pd.to_numeric(ar_6h[lakeid], errors='coerce').fillna(0).to_numpy()
    shapemap = (shapemap > 0).astype(float)
    shapemap_agg = np.nanmax(shapemap.reshape(-1, 4), axis=1)
    ar_daily_dict[lakeid] = shapemap_agg
ar_daily = pd.DataFrame(ar_daily_dict, index=ar_time_day)

ar_daily.to_csv(root / 'AR_extraction_daily.csv')
print(ar_daily.head())

             10  100  102  104  105  106  108  118  119  123  ...  xNL095  \
1950-09-01  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  1.0  ...     0.0   
1950-09-02  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...     0.0   
1950-09-03  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...     0.0   
1950-09-04  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...     0.0   
1950-09-05  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  ...     1.0   

            xNL096  xNL098  xNL099  xNL100  xNL101  xNL102  xSS01  xTAK01  \
1950-09-01     0.0     0.0     0.0     0.0     0.0     0.0    0.0     0.0   
1950-09-02     0.0     0.0     0.0     0.0     0.0     0.0    0.0     1.0   
1950-09-03     0.0     0.0     0.0     0.0     0.0     0.0    0.0     0.0   
1950-09-04     0.0     0.0     0.0     0.0     0.0     0.0    0.0     0.0   
1950-09-05     1.0     1.0     1.0     0.0     1.0     1.0    0.0     0.0   

            xTN001  
1950-09-01     0.0  
1950-09-02     0.0  
1950-09-03 

In [4]:
# Calculate annual and seasonal AR frequency for each lake based on hydrological years

ar_daily = pd.read_csv(root / 'AR_extraction_daily.csv', index_col=0)
ar_daily.index = pd.to_datetime(ar_daily.index)

hydro_year = ar_daily.index.year.copy()
hydro_year = pd.Series(hydro_year, index=ar_daily.index) 
hydro_year[ar_daily.index.month >= 9] += 1

autumn_winter_mask = (ar_daily.index.month >= 9) | (ar_daily.index.month <= 2)
spring_summer_mask = (ar_daily.index.month >= 3) & (ar_daily.index.month <= 8)

ar_annual = ar_daily.groupby(hydro_year).sum()
ar_autumn_winter = ar_daily[autumn_winter_mask].groupby(hydro_year).sum()
ar_spring_summer = ar_daily[spring_summer_mask].groupby(hydro_year).sum()

ar_annual.to_csv(root / 'AR_frequency_annual_AllYear.csv')
ar_autumn_winter.to_csv(root / 'AR_frequency_annual_AutWin.csv')
ar_spring_summer.to_csv(root / 'AR_frequency_annual_SprSum.csv')

print(ar_annual.head())

        10   100   102   104   105   106   108   118   119   123  ...  xNL095  \
1951  76.0  75.0  76.0  79.0  74.0  76.0  76.0  75.0  75.0  82.0  ...    41.0   
1952  76.0  73.0  76.0  72.0  71.0  73.0  74.0  69.0  69.0  75.0  ...    64.0   
1953  63.0  65.0  68.0  65.0  61.0  64.0  60.0  62.0  62.0  57.0  ...    49.0   
1954  65.0  65.0  60.0  63.0  65.0  67.0  65.0  67.0  67.0  63.0  ...    66.0   
1955  71.0  72.0  70.0  70.0  74.0  69.0  70.0  76.0  76.0  78.0  ...    52.0   

      xNL096  xNL098  xNL099  xNL100  xNL101  xNL102  xSS01  xTAK01  xTN001  
1951    48.0    46.0    44.0    33.0    43.0    45.0   75.0    49.0    51.0  
1952    80.0    81.0    80.0    61.0    74.0    75.0   85.0    48.0    45.0  
1953    52.0    52.0    51.0    39.0    56.0    55.0   62.0    40.0    49.0  
1954    62.0    62.0    60.0    72.0    61.0    62.0   70.0    43.0    66.0  
1955    40.0    38.0    39.0    51.0    43.0    41.0   66.0    43.0    55.0  

[5 rows x 652 columns]


In [ ]:
# Extract 6-hour IVT for each lake based on the original AR data

ar_dataset = xr.open_dataset(root / 'globalARcatalog_ERA5_1940-2024_v4.0.nc')
ivtx = np.squeeze(ar_dataset['ivtx'].to_numpy())
ivty = np.squeeze(ar_dataset['ivty'].to_numpy())
ivt = np.sqrt((ivtx ** 2) + (ivty ** 2))
del ivtx, ivty

ar_6h = pd.read_csv(root / 'AR_extraction_6h.csv', index_col=0)
results = {}

for col in ar_6h.columns:
    shapemap = ar_6h[col].values
    shapemap[np.isnan(shapemap)] = 0.0

    vals = []
    for idx, i in enumerate(shapemap):
        x = ivt[idx, int(i) - 1]
        vals.append(x)

    vals = np.array(vals)
    vals[np.isnan(vals)] = 0.0

    results[col] = vals

ivt_6h = pd.DataFrame(results, index=ar_6h.index)
ivt_6h.to_csv(root / 'IVT_extraction_6h.csv')


In [25]:
# Aggregate 6-hour IVT data to daily mean IVT for each lake

ivt_6h = pd.read_csv(root / 'IVT_extraction_6h.csv', index_col=0)
ivt_6h.index = pd.to_datetime(ivt_6h.index)
ivt_6h = ivt_6h.loc['1950-09-01':'2022-08-31']
time = ivt_6h.index
time_day = np.array(time).reshape(-1, 4)[:,0]

ivt_daily_dict = {}
for lakeid in ivt_6h.columns:
    ivts = ivt_6h[lakeid].values
    ivts[np.isnan(ivts)] = 0.0
    ivts_mean = np.mean(ivts.reshape(-1, 4), axis=1)
    ivt_daily_dict[lakeid] = ivts_mean
ivt_daily = pd.DataFrame(ivt_daily_dict, index=time_day)

ivt_daily.to_csv(root / 'IVT_extraction_daily.csv')
print(ivt_daily.head())

                    10         100         102         104         105  \
1950-09-01  176.021925  277.498357  176.021925  277.498357  277.498357   
1950-09-02    0.000000    0.000000    0.000000    0.000000    0.000000   
1950-09-03    0.000000    0.000000    0.000000    0.000000    0.000000   
1950-09-04    0.000000    0.000000    0.000000    0.000000    0.000000   
1950-09-05    0.000000    0.000000    0.000000    0.000000    0.000000   

                   106         108         118         119         123  ...  \
1950-09-01  277.498357  277.498357  277.498357  277.498357  277.498357  ...   
1950-09-02    0.000000    0.000000    0.000000    0.000000    0.000000  ...   
1950-09-03    0.000000    0.000000    0.000000    0.000000    0.000000  ...   
1950-09-04    0.000000    0.000000    0.000000    0.000000    0.000000  ...   
1950-09-05    0.000000    0.000000    0.000000    0.000000    0.000000  ...   

             xNL095     xNL096     xNL098     xNL099  xNL100     xNL101  \
1950-